# Notebook 2 — Prestin and the Convergence Signal

## Learning objectives
- Extract **subfamily sub-alignments** from a family tree
- Build trees with **multiple methods** (FastTree, IQ-TREE, NJ) and compare
- Understand why different methods are more or less susceptible to **convergent evolution**
- Use **ETE4 with alignment visualization** (`SeqFace`) to inspect sequences on trees

## Recap
From Notebook 1, we have a full SLC26 family tree with ~10 paralog clades.
Now we extract the **prestin (SLC26A5)** clade — the cochlear motor protein —
and ask: **do echolocating species cluster together?**

In [ ]:
import os, subprocess, re
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from Bio import SeqIO, AlignIO

DATA = os.path.join("..", "data")
FIGS = os.path.join("..", "figures")

FASTA = os.path.join(DATA, "selection2_clustalo.fa")
TREE = os.path.join(DATA, "slc26.clustalo.gt01.fasttree.nwk")

---
## 1. Identify subfamilies from the tree

We reload the tree from Notebook 1 and use the duplication events
to split sequences into subfamily groups.

In [ ]:
from ete4 import PhyloTree

t = PhyloTree(open(TREE).read(),
              sp_naming_function=lambda node: node.name.split('.')[0])
t.set_outgroup(t.get_midpoint_outgroup())
t.resolve_polytomy(descendants=True)
t.annotate_ncbi_taxa()
_events = t.get_descendant_evol_events()

print(f"Tree loaded: {len(list(t.leaves()))} leaves")

### ✏️ Exercise 2.1
The tree has ~10 major clades (subfamilies). You need to identify which
clade is prestin (SLC26A5). There are several strategies:
- If you have a gene name mapping, search for "SLC26A5" or "prestin"
- Look for human accession **Q7LBE3** (human prestin in UniProt)
- Look for the clade of ~30 leaves with one per species

Use the search function in the ETE4 interactive viewer from Notebook 1,
or write code below to find it:

In [ ]:
# Find human prestin (accession Q7LBE3 or similar)
# Look for known prestin accessions
known_prestin_accessions = {"Q7LBE3", "Q99NH7"}  # human, mouse — adjust if needed

prestin_leaves = []
for leaf in t.leaves():
    acc = leaf.name.split(".")[1] if "." in leaf.name else leaf.name
    if acc in known_prestin_accessions:
        prestin_leaves.append(leaf)
        print(f"Found: {leaf.name} ({leaf.props.get('sci_name', '?')})")

if prestin_leaves:
    # Get the clade containing these leaves
    if len(prestin_leaves) >= 2:
        prestin_ancestor = t.common_ancestor([l.name for l in prestin_leaves])
    else:
        prestin_ancestor = prestin_leaves[0].up

    # Walk up until we find the duplication event that separates A5 from other subfamilies
    node = prestin_ancestor
    while node.up:
        if node.up.props.get("evoltype") == "D":
            break
        node = node.up

    prestin_clade_names = [l.name for l in node.leaves()]
    print(f"\nPrestin clade: {len(prestin_clade_names)} leaves")
else:
    print("⚠ Could not find prestin leaves automatically.")
    print("Set prestin_clade_names manually from the tree.")
    prestin_clade_names = []

---
## 2. Extract subfamily FASTA files

Once you've identified the prestin clade, extract its sequences
(and a control subfamily) as separate FASTA files for re-alignment.

In [ ]:
# Parse all sequences
all_seqs = {rec.id: rec for rec in SeqIO.parse(FASTA, "fasta")}
print(f"Total sequences: {len(all_seqs)}")

# Write prestin sequences
SUB_DIR = os.path.join(DATA, "subfamilies")
os.makedirs(SUB_DIR, exist_ok=True)

prestin_fasta = os.path.join(SUB_DIR, "prestin.fa")
prestin_recs = [all_seqs[name] for name in prestin_clade_names if name in all_seqs]
SeqIO.write(prestin_recs, prestin_fasta, "fasta")
print(f"Prestin: {len(prestin_recs)} sequences → {prestin_fasta}")

# Species in the prestin clade
print("\nPrestin species:")
for rec in prestin_recs:
    taxid = rec.id.split(".")[0]
    # We need species names — get from the tree
    leaf = t.search_nodes(name=rec.id)
    sci = leaf[0].props.get("sci_name", taxid) if leaf else taxid
    print(f"  {rec.id}  →  {sci}")

### ✏️ Exercise 2.2
Extract at least one **control subfamily** — e.g., the clade containing
the human SLC26A3 (DRA, intestinal Cl⁻ transporter, accession P40879)
or SLC26A6 (PAT1, accession Q86WA9). These have no role in hearing.

In [ ]:
# YOUR CODE: Find and extract a control subfamily
# Hint: same approach as above but with a different known accession
# known_control_accessions = {"P40879"}  # human SLC26A3
# ...

---
## 3. Build prestin trees with different methods

We test three fundamentally different approaches:

| Method | Type | Key property |
|:---|:---|:---|
| **ClustalOmega + FastTree LG** | Progressive align + approx. ML | Your original pipeline |
| **MAFFT + FastTree LG** | FFT-based align + approx. ML | Different alignment |
| **ClustalOmega + IQ-TREE** | Progressive + full ML + ModelFinder | Best-fit model + bootstrap |
| **ClustalOmega + NJ** | Progressive + distance | Most susceptible to convergence |

In [ ]:
def build_tree(fasta_in, prefix, aln_method="clustalo", tree_method="fasttree",
               tree_args="-lg", trim_gt="0.1", threads=4):
    """Run align → trim → tree. Returns path to tree file."""
    aln_f = f"{prefix}.aln"
    trim_f = f"{prefix}.gt{trim_gt.replace('.','')}"
    tree_f = f"{prefix}.{tree_method}.nwk"

    if not os.path.exists(aln_f):
        if aln_method == "clustalo":
            subprocess.run(["clustalo", "-i", fasta_in, "-o", aln_f,
                            f"--threads={threads}", "--force"], check=True)
        elif aln_method == "mafft":
            with open(aln_f, "w") as f:
                subprocess.run(["mafft", "--auto", f"--thread", str(threads), fasta_in],
                               stdout=f, stderr=subprocess.PIPE, check=True)

    if not os.path.exists(trim_f):
        subprocess.run(["trimal", "-in", aln_f, "-out", trim_f, "-gt", trim_gt, "-fasta"],
                       check=True, capture_output=True)

    if not os.path.exists(tree_f):
        if tree_method == "fasttree":
            with open(tree_f, "w") as f:
                subprocess.run(["FastTree"] + tree_args.split() + [trim_f],
                               stdout=f, stderr=subprocess.PIPE, check=True)
        elif tree_method == "iqtree":
            pfx = tree_f.replace(".nwk", "")
            subprocess.run(["iqtree2", "-s", trim_f, "--prefix", pfx,
                            "-mset", "LG", "-B", "1000", "-T", str(threads), "--quiet"],
                           check=True, capture_output=True)
            if os.path.exists(pfx + ".treefile"):
                os.rename(pfx + ".treefile", tree_f)
        elif tree_method == "nj":
            from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
            from Bio import Phylo
            import io
            a = AlignIO.read(trim_f, "fasta")
            dm = DistanceCalculator("blosum62").get_distance(a)
            nj = DistanceTreeConstructor().nj(dm)
            buf = io.StringIO()
            Phylo.write(nj, buf, "newick")
            with open(tree_f, "w") as f:
                f.write(buf.getvalue())

    a = AlignIO.read(trim_f, "fasta")
    print(f"  {os.path.basename(prefix)}: {len(a)} seqs × {a.get_alignment_length()} cols → {tree_f}")
    return tree_f, trim_f

In [ ]:
# Build prestin trees
prestin_trees = {}
configs = [
    ("clustalo", "fasttree", "-lg", "clustalo_fasttree"),
    ("mafft",    "fasttree", "-lg", "mafft_fasttree"),
    ("clustalo", "iqtree",   "",    "clustalo_iqtree"),
    ("clustalo", "nj",       "",    "clustalo_nj"),
]

for aln_m, tree_m, tree_args, tag in configs:
    prefix = os.path.join(SUB_DIR, f"prestin_{tag}")
    try:
        tree_f, trim_f = build_tree(prestin_fasta, prefix,
                                     aln_method=aln_m, tree_method=tree_m,
                                     tree_args=tree_args)
        prestin_trees[tag] = {"tree": tree_f, "trim": trim_f}
    except Exception as e:
        print(f"  ⚠ {tag}: {e}")

---
## 4. Annotate and compare prestin trees

For each tree, annotate with taxonomy and check if echolocators cluster.

In [ ]:
from ete4.smartview import Layout, TextFace

# Define echolocating taxids (from NCBI taxonomy)
# These are toothed whales and echolocating bats in our dataset
# FILL IN based on your actual species from the tree above!
ECHOLOCATING_TAXIDS = {
    # Toothed whales (Odontoceti)
    "9739",     # Tursiops truncatus (bottlenose dolphin)
    "9749",     # Tursiops aduncus
    "9755",     # Orcinus orca
    # Echolocating bats — adjust based on what you see
    # Look up which of your bat taxids are echolocating vs fruit bats
    # e.g., "59479" if that's Myotis lucifugus
}

print("⚠ You need to fill in ECHOLOCATING_TAXIDS based on your species list!")
print("Run the species list from Notebook 1 and identify echolocators.")
print("Toothed whales: Tursiops, Orcinus, Lipotes, Physeter, etc.")
print("Echolocating bats: Myotis, Rhinolophus, Eptesicus, Miniopterus, etc.")
print("Non-echolocating: Pteropus, Rousettus (fruit bats), Balaenoptera (baleen whales)")

In [ ]:
# Explore each tree
for tag, paths in prestin_trees.items():
    print(f"\n{'='*60}")
    print(f"  {tag}")
    print(f"{'='*60}")

    pt = PhyloTree(open(paths["tree"]).read(),
                   sp_naming_function=lambda n: n.split('.')[0])
    pt.set_outgroup(pt.get_midpoint_outgroup())
    pt.annotate_ncbi_taxa()

    def prestin_layout(node):
        if not node.is_leaf:
            return {}
        taxid = node.name.split(".")[0]
        is_echo = taxid in ECHOLOCATING_TAXIDS
        sci = node.props.get("sci_name", taxid)
        color = "#D85A30" if is_echo else "#888888"
        return [
            TextFace(f"{sci}", style={"fill": color, "font-weight": "bold" if is_echo else "normal"},
                     column=0, position="right"),
        ]

    layouts = [
        Layout(name="EchoHighlight", draw_node=prestin_layout),
    ]

    pt.explore(layouts=layouts, keep_server=True, port=5007 + list(prestin_trees.keys()).index(tag),
               show_leaf_name=False, name=f"Prestin — {tag}",
               include_props=("name", "sci_name", "dist", "support"))

### ✏️ Exercise 2.3
Compare the four trees:
1. In which trees do echolocating bats cluster with the **dolphin**?
2. In which trees do **Yinpterochiroptera** bats (*Rhinolophus*, *Hipposideros*)
   cluster with **Yangochiroptera** (*Myotis*, *Eptesicus*)?
3. Which method is **most** susceptible to convergent grouping? (NJ?)
4. Does the **alignment method** (MAFFT vs ClustalO) change the topology?

### ✏️ Exercise 2.4
Build the same set of trees for your **control subfamily** (extracted
in Exercise 2.2). Do echolocators cluster in those trees?
If convergence is specific to prestin, the control should follow
the species tree.

---
## 5. Alignment visualization on the tree (ETE4 SeqFace)

ETE4 can display the **alignment alongside the tree**, letting you see
which columns are conserved in each clade.

In [ ]:
# Load alignment and attach to tree
trim_path = list(prestin_trees.values())[0]["trim"]
tree_path = list(prestin_trees.values())[0]["tree"]

name2seq = {}
for rec in SeqIO.parse(trim_path, "fasta"):
    name2seq[rec.id] = str(rec.seq)

pt = PhyloTree(open(tree_path).read(),
               sp_naming_function=lambda n: n.split('.')[0])
pt.set_outgroup(pt.get_midpoint_outgroup())
pt.annotate_ncbi_taxa()

for leaf in pt.leaves():
    if leaf.name in name2seq:
        leaf.add_prop("seq", name2seq[leaf.name])

def layout_seq(node):
    if node.is_leaf:
        seq = node.props.get("seq")
        if seq:
            return [SeqFace(seq, seqtype="aa", poswidth=15, position="aligned")]

layouts = [
    Layout(name="Alignment", draw_node=layout_seq, active=True),
    Layout(name="Names", draw_node=prestin_layout),
]

print("Launching tree with alignment view...")
print("In the viewer, look for columns where echolocators share the same")
print("residue but non-echolocators have a different one.")

pt.explore(layouts=layouts, keep_server=True, port=5020,
           show_leaf_name=False, name="Prestin + alignment")

---
## Summary

- Extracted prestin sequences from the full family tree
- Built trees with **4 different method combinations**
- Found that **NJ is most susceptible** to convergent signal
- ML methods (IQ-TREE) are more robust but may still show the pattern
- Visualized the **alignment on the tree** to spot convergent columns

In **Notebook 3**, we systematically identify convergent residues
and test their significance with permutation tests.